In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

bronze = spark.table("workspace.default.bronze_consumption")
print(bronze.count())
bronze.printSchema()

8400
root
 |-- consumptionGroup: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- lastUpdatedTime: string (nullable = true)
 |-- loaded_at: string (nullable = true)
 |-- meteringPointCount: long (nullable = true)
 |-- priceArea: string (nullable = true)
 |-- quantityKwh: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- startTime: string (nullable = true)



In [0]:
silver = (
    bronze
    .select(
        F.to_timestamp("startTime").alias("start_utc"),
        F.to_timestamp("endTime").alias("end_utc"),
        F.col("priceArea").alias("area"),
        F.col("consumptionGroup").alias("consumption_group"),
        F.col("quantityKwh").alias("quantity_kwh"),
        F.col("meteringPointCount").alias("metering_points"),
        F.to_timestamp("lastUpdatedTime").alias("last_updated_utc"),
        "source_file",
    )
    .dropDuplicates(["area", "consumption_group", "start_utc"])
)

print(silver.count())

8400


In [0]:
silver.orderBy("start_utc", "area", "consumption_group").show(5, truncate=False)

+-------------------+-------------------+----+-----------------+------------+---------------+-------------------+--------------------------------------------------+
|start_utc          |end_utc            |area|consumption_group|quantity_kwh|metering_points|last_updated_utc   |source_file                                       |
+-------------------+-------------------+----+-----------------+------------+---------------+-------------------+--------------------------------------------------+
|2026-10-04 22:00:00|2026-10-04 23:00:00|NO1 |cabin            |57659.516   |110856         |2026-10-07 15:04:15|consumption/2026-10-05/consumption_2026-10-05.json|
|2026-10-04 22:00:00|2026-10-04 23:00:00|NO1 |household        |1397996.8   |1122971        |2026-10-07 15:04:15|consumption/2026-10-05/consumption_2026-10-05.json|
|2026-10-04 22:00:00|2026-10-04 23:00:00|NO1 |primary          |30596.31    |7450           |2026-10-07 15:04:15|consumption/2026-10-05/consumption_2026-10-05.json|
|2026-10-0

In [0]:
silver.write.mode("overwrite").saveAsTable("workspace.default.silver_consumption")
print(spark.table("workspace.default.silver_consumption").count())

8400
